# PC1 - Detección de contradicciones normativas


## 0. Instalación y directorio de trabajo

In [1]:
!python -m pip install -q --break-system-packages torch "transformers>=4.44" jsonschema

In [2]:
import os

if os.path.basename(os.getcwd()) == 'notebook':
    os.chdir('..')
print('cwd:', os.getcwd())

cwd: /workspace/CC-0F4/Evaluaciones/CC-0F4/cc0f4-pc1-unda-contradicciones


## 1. Validador de salida (JSON parseable / schema-valid / predicción)

In [3]:
import json, re
from jsonschema import validate, ValidationError

SCHEMA = json.load(open('data/schema.json', encoding='utf-8'))

def extract_json(text):
    m = re.search(r'\{.*\}', text, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

def check(raw):
    obj = extract_json(raw)
    if obj is None:
        return False, False, None, None
    try:
        validate(obj, SCHEMA)
        return True, True, obj['label'], obj
    except ValidationError:
        pred = obj.get('label') if isinstance(obj, dict) else None
        return True, False, pred, obj

check('{"label": "neutral", "confidence": 0.5, "reason": "ok"}')

(True,
 True,
 'neutral',
 {'label': 'neutral', 'confidence': 0.5, 'reason': 'ok'})

## 2. Cargar el modelo

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
SYSTEM = 'Eres un asistente que analiza textos normativos peruanos.'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype='auto').to(device)
model.eval()
print('modelo cargado')

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cpu


Loading weights: 100%|██████████| 290/290 [00:01<00:00, 284.16it/s]


modelo cargado


In [5]:
cases = [json.loads(l) for l in open('data/cases.jsonl', encoding='utf-8') if l.strip()]
len(cases), cases[0]

(9,
 {'id': 1,
  'norma_a': 'Los contribuyentes deberán presentar la declaración jurada dentro de los diez días hábiles siguientes al cierre del mes.',
  'norma_b': 'Los contribuyentes podrán presentar la declaración jurada hasta treinta días calendario después del cierre del mes.',
  'gold': 'contradiction',
  'origen': 'sintético'})

## 3. Función para correr una condición experimental

In [6]:
def run_condition(prompt_path, context_path, decoding='greedy',
                   temperature=0.8, top_p=0.9, top_k=50, seed=42, n_runs=1,
                   max_new_tokens=150, out_path=None):
    template = open(prompt_path, encoding='utf-8').read()
    contexto = open(context_path, encoding='utf-8').read().strip()
    config = {'model': MODEL, 'prompt': prompt_path, 'context': context_path,
              'decoding': decoding, 'max_new_tokens': max_new_tokens}
    if decoding == 'greedy':
        gen_kw = dict(do_sample=False, temperature=None, top_p=None, top_k=None)
        n_runs = 1
    else:
        gen_kw = dict(do_sample=True, temperature=temperature, top_p=top_p, top_k=top_k)
        config.update(temperature=temperature, top_p=top_p, top_k=top_k, seed=seed, n_runs=n_runs)
    rows = [{'config': config}]
    for c in cases:
        user = template.format(contexto=contexto, norma_a=c['norma_a'], norma_b=c['norma_b'])
        msgs = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors='pt').to(device)
        for r in range(n_runs):
            if decoding == 'sampling':
                set_seed(seed + r)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=max_new_tokens, **gen_kw)
            raw = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            parseable, valid, pred, obj = check(raw)
            rows.append({'id': c['id'], 'run': r, 'gold': c['gold'], 'pred': pred,
                         'json_parseable': parseable, 'schema_valid': valid, 'raw': raw})
            print(c['id'], r, pred, valid)
    if out_path:
        with open(out_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(json.dumps(x, ensure_ascii=False) for x in rows))
    return rows

## Experimento 1 — Prompt baseline vs mejorado (contexto C0, greedy)

In [7]:
rows_P0 = run_condition('prompts/baseline.txt', 'prompts/context_C0.txt', decoding='greedy',
                          out_path='results/exp1_P0.jsonl')

1 0 neutral True
2 0 neutral False
3 0 neutral True
4 0 neutral True
5 0 neutral True
6 0 neutral True
7 0 neutral False
8 0 neutral True
9 0 neutral True


In [8]:
rows_P1 = run_condition('prompts/mejorado.txt', 'prompts/context_C0.txt', decoding='greedy',
                          out_path='results/exp1_P1.jsonl')

1 0 neutral True
2 0 neutral True
3 0 neutral False
4 0 neutral True
5 0 neutral True
6 0 neutral True
7 0 neutral False
8 0 neutral True
9 0 neutral True


## Experimento 2 — Contexto neutral vs conflictivo (prompt P1, greedy)

In [9]:
rows_C1 = run_condition('prompts/mejorado.txt', 'prompts/context_C1.txt', decoding='greedy',
                          out_path='results/exp2_C1.jsonl')

1 0 neutral True
2 0 neutral True
3 0 neutral True
4 0 neutral True
5 0 neutral True
6 0 neutral True
7 0 neutral True
8 0 neutral False
9 0 neutral True


## Experimento 3 — Decoding: greedy (=exp1_P1) vs sampling, 5 corridas

In [10]:
rows_D1 = run_condition('prompts/mejorado.txt', 'prompts/context_C0.txt', decoding='sampling',
                          temperature=0.8, top_p=0.9, top_k=50, seed=42, n_runs=5,
                          out_path='results/exp3_D1.jsonl')

1 0 neutral True
1 1 contradiction True
1 2 contradiction True
1 3 neutral True
1 4 None False
2 0 neutral True
2 1 contradiction True
2 2 neutral True
2 3 neutral True
2 4 neutral True
3 0 neutral True
3 1 contradiction True
3 2 neutral True
3 3 neutral True
3 4 neutral True
4 0 neutral True
4 1 contradiction True
4 2 contradiction True
4 3 neutral True
4 4 contradiction True
5 0 neutral True
5 1 contradiction True
5 2 neutral True
5 3 neutral True
5 4 entailment True
6 0 neutral True
6 1 contradiction True
6 2 neutral True
6 3 neutral True
6 4 neutral True
7 0 neutral True
7 1 contradiction True
7 2 neutral True
7 3 neutral True
7 4 entailment True
9 0 neutral True
9 1 contradiction True
9 2 neutral True
9 3 neutral True
9 4 entailment True


## 4. Métricas


In [11]:
from collections import Counter, defaultdict

LABELS = ['contradiction', 'entailment', 'neutral']

def load_jsonl(path):
    rows = [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]
    return rows[0]['config'], rows[1:]

def macro_f1(rows):
    f1s = []
    for lab in LABELS:
        tp = sum(r['pred'] == lab and r['gold'] == lab for r in rows)
        fp = sum(r['pred'] == lab and r['gold'] != lab for r in rows)
        fn = sum(r['pred'] != lab and r['gold'] == lab for r in rows)
        p = tp / (tp + fp) if tp + fp else 0
        rc = tp / (tp + fn) if tp + fn else 0
        f1s.append(2 * p * rc / (p + rc) if p + rc else 0)
    return sum(f1s) / len(f1s)

def summarize(path):
    cfg, rows = load_jsonl(path)
    n = len(rows)
    for r in rows:  # solo cuentan como predicción las salidas schema-valid
        if not r['schema_valid']:
            r['pred'] = None
    s = {
        'archivo': path,
        'n': n,
        'json_parseable_rate': round(sum(r['json_parseable'] for r in rows) / n, 3),
        'schema_valid_rate': round(sum(r['schema_valid'] for r in rows) / n, 3),
        'accuracy': round(sum(r['pred'] == r['gold'] for r in rows) / n, 3),
        'macro_f1': round(macro_f1(rows), 3),
        'distribucion_pred': dict(Counter(r['pred'] for r in rows)),
    }
    by_case = defaultdict(list)
    for r in rows:
        by_case[r['id']].append(r['pred'])
    if max(len(v) for v in by_case.values()) > 1:
        stab = {i: round(Counter(v).most_common(1)[0][1] / len(v), 2) for i, v in by_case.items()}
        s['estabilidad_etiqueta_por_caso'] = stab
        s['estabilidad_media'] = round(sum(stab.values()) / len(stab), 3)
    return cfg, s

In [12]:
for path in ['results/exp1_P0.jsonl', 'results/exp1_P1.jsonl', 'results/exp2_C1.jsonl', 'results/exp3_D1.jsonl']:
    cfg, s = summarize(path)
    print(json.dumps({'config': cfg, 'metricas': s}, ensure_ascii=False, indent=2))

{
  "config": {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "prompt": "prompts/baseline.txt",
    "context": "prompts/context_C0.txt",
    "decoding": "greedy",
    "max_new_tokens": 150
  },
  "metricas": {
    "archivo": "results/exp1_P0.jsonl",
    "n": 9,
    "json_parseable_rate": 1.0,
    "schema_valid_rate": 0.778,
    "accuracy": 0.222,
    "macro_f1": 0.133,
    "distribucion_pred": {
      "neutral": 7,
      "null": 2
    }
  }
}
{
  "config": {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "prompt": "prompts/mejorado.txt",
    "context": "prompts/context_C0.txt",
    "decoding": "greedy",
    "max_new_tokens": 150
  },
  "metricas": {
    "archivo": "results/exp1_P1.jsonl",
    "n": 9,
    "json_parseable_rate": 1.0,
    "schema_valid_rate": 0.778,
    "accuracy": 0.222,
    "macro_f1": 0.133,
    "distribucion_pred": {
      "neutral": 7,
      "null": 2
    }
  }
}
{
  "config": {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "prompt": "prompts/mejorado.txt",
    "co